In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 19


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 3.8266801685094833
Epoch 2/100, Loss: 3.9017274528741837
Epoch 3/100, Loss: 3.8683845549821854
Epoch 4/100, Loss: 3.8324596285820007
Epoch 5/100, Loss: 3.8568407744169235
Epoch 6/100, Loss: 3.5690263360738754
Epoch 7/100, Loss: 3.2194144129753113
Epoch 8/100, Loss: 3.4425197392702103
Epoch 9/100, Loss: 3.7947756499052048
Epoch 10/100, Loss: 3.910467177629471
Epoch 11/100, Loss: 3.816099099814892


Epoch 12/100, Loss: 3.520236775279045
Epoch 13/100, Loss: 3.919036567211151
Epoch 14/100, Loss: 3.5385866686701775
Epoch 15/100, Loss: 3.6147948130965233
Epoch 16/100, Loss: 3.883748948574066
Epoch 17/100, Loss: 3.5045972988009453
Epoch 18/100, Loss: 3.4265563637018204
Epoch 19/100, Loss: 3.5542204529047012
Epoch 20/100, Loss: 3.7634765058755875
Epoch 21/100, Loss: 3.7782819718122482
Epoch 22/100, Loss: 3.8340120017528534
Epoch 23/100, Loss: 3.9229934364557266


Epoch 24/100, Loss: 3.4934798032045364
Epoch 25/100, Loss: 4.003281280398369
Epoch 26/100, Loss: 3.3680643141269684
Epoch 27/100, Loss: 3.6578859388828278
Epoch 28/100, Loss: 3.677228033542633
Epoch 29/100, Loss: 3.7755416855216026
Epoch 30/100, Loss: 3.8309423476457596
Epoch 31/100, Loss: 3.9262941405177116
Epoch 32/100, Loss: 3.540925294160843
Epoch 33/100, Loss: 3.575592577457428
Epoch 34/100, Loss: 3.8866679444909096


Epoch 35/100, Loss: 3.9675141870975494
Epoch 36/100, Loss: 3.706845134496689
Epoch 37/100, Loss: 3.8477499037981033
Epoch 38/100, Loss: 3.7034248411655426
Epoch 39/100, Loss: 3.7597203478217125
Epoch 40/100, Loss: 3.6919088661670685
Epoch 41/100, Loss: 3.698540970683098
Epoch 42/100, Loss: 3.8576692789793015
Epoch 43/100, Loss: 3.908907875418663
Epoch 44/100, Loss: 3.6543520092964172
Epoch 45/100, Loss: 3.6281170025467873


Epoch 46/100, Loss: 3.729173995554447
Epoch 47/100, Loss: 3.8575748652219772
Epoch 48/100, Loss: 3.346373163163662
Epoch 49/100, Loss: 3.2141584679484367
Epoch 50/100, Loss: 3.2252571433782578
Epoch 51/100, Loss: 3.7562811374664307
Epoch 52/100, Loss: 3.715036541223526
Epoch 53/100, Loss: 3.644934356212616
Epoch 54/100, Loss: 3.808081239461899
Epoch 55/100, Loss: 3.9533345252275467
Epoch 56/100, Loss: 3.860452100634575
Epoch 57/100, Loss: 3.5244970694184303


...

Epoch 29/100, Loss: 1.8802501857280731
Epoch 30/100, Loss: 1.881369449198246
Epoch 31/100, Loss: 1.8581175804138184
Epoch 32/100, Loss: 1.9503894224762917
Epoch 33/100, Loss: 1.9225239604711533
Epoch 34/100, Loss: 1.9219208285212517
Epoch 35/100, Loss: 1.8441737666726112
Epoch 36/100, Loss: 1.952710174024105
Epoch 37/100, Loss: 1.9097493514418602
Epoch 38/100, Loss: 1.8718701675534248
Epoch 39/100, Loss: 1.9585497677326202
Epoch 40/100, Loss: 1.8802310600876808
Epoch 41/100, Loss: 1.958050288259983
Epoch 42/100, Loss: 1.9871834516525269
Epoch 43/100, Loss: 1.9772472232580185


Epoch 44/100, Loss: 1.984911248087883
Epoch 45/100, Loss: 1.9062564186751842
Epoch 46/100, Loss: 1.90795286744833
Epoch 47/100, Loss: 1.933620236814022
Epoch 48/100, Loss: 1.9275238364934921
Epoch 49/100, Loss: 2.013308972120285
Epoch 50/100, Loss: 1.8463918566703796
Epoch 51/100, Loss: 1.9348467886447906
Epoch 52/100, Loss: 1.879821516573429
Epoch 53/100, Loss: 1.8881775215268135
Epoch 54/100, Loss: 1.9815171658992767
Epoch 55/100, Loss: 1.8511261269450188
Epoch 56/100, Loss: 1.9740051180124283
Epoch 57/100, Loss: 1.9983704164624214
Epoch 58/100, Loss: 1.9867114797234535


Epoch 59/100, Loss: 1.9274633005261421
Epoch 60/100, Loss: 1.894498959183693
Epoch 61/100, Loss: 2.0903030410408974
Epoch 62/100, Loss: 1.9648971036076546
Epoch 63/100, Loss: 1.898339994251728
Epoch 64/100, Loss: 1.9652885347604752
Epoch 65/100, Loss: 1.9362775012850761
Epoch 66/100, Loss: 1.9415520653128624
Epoch 67/100, Loss: 1.8297626078128815
Epoch 68/100, Loss: 1.938528448343277
Epoch 69/100, Loss: 1.8626821637153625
Epoch 70/100, Loss: 1.898349568247795
Epoch 71/100, Loss: 1.878764845430851
Epoch 72/100, Loss: 1.9514448046684265
Epoch 73/100, Loss: 1.823836199939251


Epoch 74/100, Loss: 1.9564805254340172
Epoch 75/100, Loss: 1.9030485525727272
Epoch 76/100, Loss: 1.9339545369148254
Epoch 77/100, Loss: 1.8643614053726196
Epoch 78/100, Loss: 1.892007738351822
Epoch 79/100, Loss: 1.847023919224739
Epoch 80/100, Loss: 1.9675718694925308
Epoch 81/100, Loss: 1.9580830559134483
Epoch 82/100, Loss: 1.9136268347501755
Epoch 83/100, Loss: 1.9129740744829178
Epoch 84/100, Loss: 1.8149800077080727
Epoch 85/100, Loss: 1.9942160546779633
Epoch 86/100, Loss: 1.8644409403204918
Epoch 87/100, Loss: 2.0238108187913895
Epoch 88/100, Loss: 1.929457500576973


Epoch 89/100, Loss: 1.8887943550944328
Epoch 90/100, Loss: 1.9181177616119385
Epoch 91/100, Loss: 1.9310107752680779
Epoch 92/100, Loss: 1.853210136294365
Epoch 93/100, Loss: 1.8714705258607864
Epoch 94/100, Loss: 1.899749018251896
Epoch 95/100, Loss: 1.917404644191265
Epoch 96/100, Loss: 1.877234973013401
Epoch 97/100, Loss: 2.0030161887407303
Epoch 98/100, Loss: 2.0415618121623993
Epoch 99/100, Loss: 1.901619203388691
Epoch 100/100, Loss: 1.9327102228999138
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4735
